In [33]:
!pip install cantera
!pip install CoolProp
import cantera as ct
print(ct.__version__)

3.2.0


In [48]:
import os
import pandas as pd
from CoolProp.CoolProp import PropsSI

methane_mass_fraction = 1
nitrogen_mass_fraction = 0.00
FLUID = f"Methane[{methane_mass_fraction}]&Nitrogen[{nitrogen_mass_fraction}]"

def C_to_K(T_C):
    # 섭씨 온도를 켈빈 온도로 변환
    return T_C + 273.15

def K_to_C(T_K):
    # 켈빈 온도를 섭씨 온도로 변환
    return T_K - 273.15

def bar_to_Pa(P_bar):
    # bar 단위를 파스칼(Pa)로 변환
    return P_bar * 1e5

def Pa_to_bar(P_Pa):
    # 파스칼(Pa) 단위를 bar로 변환
    return P_Pa / 1e5

# =====================================================
# 1. CoolProp 속성 도우미 함수들
# =====================================================

def h_TP(T, P, fluid=FLUID):
    # 온도(T)와 압력(P)으로 질량 엔탈피(Hmass) 계산
    try:
        return PropsSI("Hmass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Hmass", "T", T + 0.1, "P", P, fluid)

def s_TP(T, P, fluid=FLUID):
    # 온도(T)와 압력(P)으로 질량 엔트로피(Smass) 계산
    try:
        return PropsSI("Smass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Smass", "T", T + 0.1, "P", P, fluid)

def T_Ph(P, h, fluid=FLUID):
    # 압력(P)과 질량 엔탈피(h)로 온도(T) 계산
    return PropsSI("T", "P", P, "Hmass", h, fluid)

def h_PS(P, s, fluid=FLUID):
    # 압력(P)과 질량 엔트로피(s)로 질량 엔탈피(Hmass) 계산 (등엔트로피 과정)
    return PropsSI("Hmass", "P", P, "Smass", s, fluid)

def rho_TP(T, P, fluid=FLUID):
    # 온도(T)와 압력(P)으로 질량 밀도(Dmass) 계산
    try:
        return PropsSI("Dmass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Dmass", "T", T + 0.1, "P", P, fluid)

# =====================================================
# 2. 컴포넌트 모델
# =====================================================

def liquid_pump(m_dot, T_in, P_in, P_out, eta=0.75):
    # 액체 펌프 모델
    h_in = h_TP(T_in, P_in)
    rho = rho_TP(T_in, P_in)

    if rho < 100.0:
        # 펌프 입구 밀도가 너무 낮으면 오류 발생 (액체가 아닌 증기/이상 상태일 가능성)
        raise ValueError(
            f"액체 펌프 입구 밀도가 너무 낮습니다 ({rho:.3f} kg/m3). "
            "입구는 증기/이상 상태일 가능성이 높습니다. "
            "펌프 입구 압력을 높이거나 펌프 입구 온도를 낮추십시오."
        )

    # 펌프 일 (엔탈피 변화)
    dh = (P_out - P_in) / rho / eta
    h_out = h_in + dh
    T_out = T_Ph(P_out, h_out)
    W = m_dot * dh

    return {
        "T_out": T_out, # 출구 온도
        "P_out": P_out, # 출구 압력
        "h_out": h_out, # 출구 엔탈피
        "W": W,         # 펌프 동력
        "rho_in": rho,  # 입구 밀도
    }

def compressor(m_dot, T_in, P_in, P_out, eta=0.75):
    # 압축기 모델
    if m_dot <= 0:
        # 질량 유량이 없으면 압축기 동력은 0
        return {
            "T_out": T_in,
            "P_out": P_out,
            "h_out": h_TP(T_in, P_out),
            "W": 0.0,
        }

    h_in = h_TP(T_in, P_in)
    s_in = s_TP(T_in, P_in)
    # 등엔트로피 출구 엔탈피
    h_out_s = h_PS(P_out, s_in)
    # 실제 출구 엔탈피 (효율 반영)
    h_out = h_in + (h_out_s - h_in) / eta
    T_out = T_Ph(P_out, h_out)
    # 압축기 동력
    W = m_dot * (h_out - h_in)

    return {
        "T_out": T_out,
        "P_out": P_out,
        "h_out": h_out,
        "W": W,
    }

def heat_exchanger_effectiveness(
    m_cold,
    T_cold_in,
    P_cold,
    m_hot,
    T_hot_in,
    P_hot,
    effectiveness=0.60,
    min_delta_T=3.0,
):
    """
    일반적인 열교환기 유용도(effectiveness) 모델.
    뜨거운 스트림이 차가운 스트림에 열을 전달합니다.
    """
    if m_hot <= 0 or effectiveness <= 0:
        # 뜨거운 유량 또는 유용도가 0 이하면 열 전달 없음
        return {
            "Q": 0.0,
            "T_cold_out": T_cold_in,
            "h_cold_out": h_TP(T_cold_in, P_cold),
            "T_hot_out": T_hot_in,
            "h_hot_out": h_TP(T_hot_in, P_hot),
        }

    h_hot_in = h_TP(T_hot_in, P_hot)
    h_cold_in = h_TP(T_cold_in, P_cold)
    # 최소 접근 온도 차이를 고려한 뜨거운 스트림의 최소 온도
    T_hot_min = T_cold_in + min_delta_T
    h_hot_min = h_TP(T_hot_min, P_hot)

    # 최대 가능한 열량 (뜨거운 스트림 기준)
    Q_max_hot = m_hot * max(h_hot_in - h_hot_min, 0.0)
    # 실제 열 전달량 (유용도 반영)
    Q = effectiveness * Q_max_hot

    # 출구 엔탈피 계산
    h_hot_out = h_hot_in - Q / m_hot
    h_cold_out = h_cold_in + Q / m_cold

    # 출구 온도 계산
    T_hot_out = T_Ph(P_hot, h_hot_out)
    T_cold_out = T_Ph(P_cold, h_cold_out)

    return {
        "Q": Q,                             # 열 전달량
        "T_cold_out": T_cold_out,         # 차가운 스트림 출구 온도
        "h_cold_out": h_cold_out,         # 차가운 스트림 출구 엔탈피
        "T_hot_out": T_hot_out,           # 뜨거운 스트림 출구 온도
        "h_hot_out": h_hot_out,           # 뜨거운 스트림 출구 엔탈피
    }

def set_stream_temperature(m_dot, T_in, P, T_target):
    """
    히터/쿨러를 적용하여 동일 압력에서 스트림 온도를 설정합니다.
    양의 Q는 스트림에 열이 추가됨을 의미합니다.
    """
    h_in = h_TP(T_in, P)
    h_out = h_TP(T_target, P)
    # 필요한 열량
    Q = m_dot * (h_out - h_in)

    return {
        "Q": Q,
        "T_out": T_target,
        "P_out": P,
        "h_out": h_out,
    }

def mix_streams_same_pressure(streams, P_mix):
    """
    동일 압력에서 여러 스트림을 엔탈피 균형을 사용하여 혼합합니다.
    각 스트림 딕셔너리는 m_dot, T, P를 포함해야 합니다.

    이 간소화된 모델에서는 P_mix와 스트림 압력이 다른 경우, 압력 일치를 등엔탈피로 처리합니다.
    이는 재액화 반환 라인에 대한 모델링 가정입니다.
    """
    total_m = sum(s["m_dot"] for s in streams)
    if total_m <= 0:
        raise ValueError("총 혼합 질량 유량은 양수여야 합니다.")

    total_h_flow = 0.0
    for s in streams:
        h = h_TP(s["T"], s["P"])
        total_h_flow += s["m_dot"] * h

    # 혼합물의 엔탈피와 온도 계산
    h_mix = total_h_flow / total_m
    T_mix = T_Ph(P_mix, h_mix)

    return {
        "m_dot": total_m,
        "T": T_mix,
        "P": P_mix,
        "h": h_mix,
    }

def vaporizer_for_mixed_sendout(
    m_ng,
    T_ng_in,
    P_ng_in,
    m_bog,
    T_bog_to_mixer,
    P_bog_to_mixer,
    T_mix_target,
    P_mix,
):
    """
    엔탈피 혼합 모델.
    기화기 출구의 NG + BOG는 스로틀링 후 목표 T/P에서 최종 혼합 가스를 생성해야 합니다.
    """
    h_ng_in = h_TP(T_ng_in, P_ng_in)

    # BOG 스로틀링 (등엔탈피 가정)
    h_bog_before_valve = h_TP(T_bog_to_mixer, P_bog_to_mixer)
    h_bog_mix = h_bog_before_valve
    T_bog_after_valve = T_Ph(P_mix, h_bog_mix)

    h_mix_target = h_TP(T_mix_target, P_mix)
    # 기화기 출구 NG의 엔탈피 계산
    h_ng_vap_out = ((m_ng + m_bog) * h_mix_target - m_bog * h_bog_mix) / m_ng
    T_ng_vap_out = T_Ph(P_mix, h_ng_vap_out)
    # 기화기 열량
    Q = m_ng * (h_ng_vap_out - h_ng_in)

    return {
        "Q": Q,
        "T_ng_vap_out": T_ng_vap_out,
        "h_ng_vap_out": h_ng_vap_out,
        "T_bog_after_valve": T_bog_after_valve,
        "T_mix_out": T_mix_target,
        "P_mix_out": P_mix,
    }


# =====================================================
# 3. 다단 HPC 모델
# =====================================================

def get_hpc_stage_pressures(P_in, P_out, n_stages):
    """
    균등 압력비 단계.
    경계 압력: [P0, P1, ..., Pn]을 반환합니다.
    """
    pr_stage = (P_out / P_in) ** (1.0 / n_stages)
    pressures = [P_in]
    for _ in range(n_stages):
        pressures.append(pressures[-1] * pr_stage)
    pressures[-1] = P_out # 마지막 압력을 P_out으로 강제 설정하여 부동 소수점 오차 방지
    return pressures


def multistage_hpc_no_intercooling(m_dot, T_in, P_in, P_out, eta=0.75, n_stages=3):
    # 중간 냉각 없는 다단 고압 압축기(HPC) 모델
    pressures = get_hpc_stage_pressures(P_in, P_out, n_stages)
    T = T_in
    W_total = 0.0
    stage_results = []

    for i in range(n_stages):
        T_stage_in = T
        comp = compressor(m_dot, T_stage_in, pressures[i], pressures[i + 1], eta)
        W_total += comp["W"]
        T = comp["T_out"]

        stage_results.append({
            "stage": i + 1,
            "P_in": pressures[i],
            "P_out": pressures[i + 1],
            "T_in": T_stage_in,
            "T_out": comp["T_out"],
            "W": comp["W"],
        })

    return {
        "T_out": T,
        "P_out": P_out,
        "W": W_total,
        "stages": stage_results,
    }


# =====================================================
# 4. 전체 공정 모델
# =====================================================

def _calculate_lng_stream(params, m_lng, P_lng_tank, T_lng_tank, P_lng_high, T_lng_pump_out_initial, m_bog_hpc, T_bog_comp_out, P_bog_comp_out):
    # LNG 펌프
    pump = liquid_pump(
        m_dot=m_lng,
        T_in=T_lng_tank,
        P_in=P_lng_tank,
        P_out=P_lng_high,
        eta=params["pump_eta"],
    )

    T_lng_after_pump_calc = pump["T_out"]
    P_lng_after_pump = pump["P_out"]

    # 펌프 출구 초기 가열
    pump_out_heater = set_stream_temperature(
        m_dot=m_lng,
        T_in=T_lng_after_pump_calc,
        P=P_lng_after_pump,
        T_target=T_lng_pump_out_initial,
    )
    T_lng_initial_heated = pump_out_heater["T_out"]
    Q_lng_pump_out_heating = pump_out_heater["Q"]

    # BOG 냉각용 디슈퍼히터 (LNG 스트림 이용)
    desuperheater = heat_exchanger_effectiveness(
        m_cold=m_lng,
        T_cold_in=T_lng_initial_heated,
        P_cold=P_lng_after_pump,
        m_hot=m_bog_hpc,
        T_hot_in=T_bog_comp_out,
        P_hot=P_bog_comp_out,
        effectiveness=params["desuperheater_effectiveness"]
    )

    T_lng_after_desuperheater = desuperheater["T_cold_out"]
    T_bog_hpc_in_after_desuperheater = desuperheater["T_hot_out"]
    Q_desuperheater = desuperheater["Q"]

    return {
        "pump": pump,
        "T_lng_after_pump_calc": T_lng_after_pump_calc,
        "P_lng_after_pump": P_lng_after_pump,
        "Q_lng_pump_out_heating": Q_lng_pump_out_heating,
        "T_lng_after_desuperheater": T_lng_after_desuperheater,
        "T_bog_hpc_in_after_desuperheater": T_bog_hpc_in_after_desuperheater,
        "Q_desuperheater": Q_desuperheater,
    }

def _calculate_bog_streams(params, m_bog_total, m_bog_hpc, P_bog_tank, T_bog_tank, P_bog_comp_out, P_bog_hpc_out, T_bog_hpc_in_after_desuperheater):
    # BOG 압축기
    comp_bog = compressor(
        m_dot=m_bog_total,
        T_in=T_bog_tank,
        P_in=P_bog_tank,
        P_out=P_bog_comp_out,
        eta=params["comp_eta"],
    )

    T_bog_comp_out = comp_bog["T_out"]
    P_bog_comp_out = comp_bog["P_out"]

    # HPC 라인
    hpc = multistage_hpc_no_intercooling(
        m_dot=m_bog_hpc,
        T_in=T_bog_hpc_in_after_desuperheater,
        P_in=P_bog_comp_out,
        P_out=P_bog_hpc_out,
        eta=params["comp_eta"],
        n_stages=3,
    )
    W_hpc = hpc["W"]

    return {
        "comp_bog": comp_bog,
        "T_bog_comp_out": T_bog_comp_out,
        "P_bog_comp_out": P_bog_comp_out,
        "hpc": hpc,
        "W_hpc": W_hpc,
    }

def _calculate_mixing_and_vaporization(params, m_lng, m_bog_hpc, m_bog_pc, P_lng_after_pump, T_lng_after_desuperheater, T_bog_reliq_return, hpc_T_out, hpc_P_out):
    # 펌프 출구 가열 및 디슈퍼히터 후의 LNG 스트림
    m_vaporizer_feed = m_lng
    T_vaporizer_feed_initial = T_lng_after_desuperheater
    P_vaporizer_feed_initial = P_lng_after_pump

    # BOG 재액화 분기 스트림이 LNG 라인으로 반환됨
    reliq_mix = mix_streams_same_pressure(
        streams=[
            {"m_dot": m_lng, "T": T_vaporizer_feed_initial, "P": P_vaporizer_feed_initial},
            {"m_dot": m_bog_pc, "T": T_bog_reliq_return, "P": P_lng_after_pump},
        ],
        P_mix=P_lng_after_pump,
    )

    T_bog_precooler_out = T_bog_reliq_return # 이전 출력 이름과의 일관성을 위해
    m_vaporizer_feed = reliq_mix["m_dot"]
    T_vaporizer_feed = reliq_mix["T"]
    P_vaporizer_feed = reliq_mix["P"]
    T_after_reliq_mix = reliq_mix["T"]

    # 기화기
    vap = vaporizer_for_mixed_sendout(
        m_ng=m_vaporizer_feed,
        T_ng_in=T_vaporizer_feed,
        P_ng_in=P_vaporizer_feed,
        m_bog=m_bog_hpc,
        T_bog_to_mixer=hpc_T_out,
        P_bog_to_mixer=hpc_P_out,
        T_mix_target=params["T_sendout"],
        P_mix=params["P_sendout"],
    )

    return {
        "reliq_mix": reliq_mix,
        "T_bog_precooler_out": T_bog_precooler_out,
        "m_vaporizer_feed": m_vaporizer_feed,
        "T_vaporizer_feed": T_vaporizer_feed,
        "P_vaporizer_feed": P_vaporizer_feed,
        "T_after_reliq_mix": T_after_reliq_mix,
        "vap": vap,
    }

def _assemble_results(params, stream_data, bog_data, mixing_data):
    # 매개변수 추출
    m_lng = params["m_lng"]
    m_bog_total = params["m_bog_total"]
    bog_to_hpc_frac = params["bog_to_hpc_frac"]

    m_bog_hpc = m_bog_total * bog_to_hpc_frac
    m_bog_pc = m_bog_total - m_bog_hpc

    P_sendout = params["P_sendout"]

    # LNG 스트림 결과
    pump = stream_data["pump"]
    T_lng_after_pump_calc = stream_data["T_lng_after_pump_calc"]
    P_lng_after_pump = stream_data["P_lng_after_pump"]
    Q_lng_pump_out_heating = stream_data["Q_lng_pump_out_heating"]
    T_lng_after_desuperheater = stream_data["T_lng_after_desuperheater"]
    T_bog_hpc_in_after_desuperheater = stream_data["T_bog_hpc_in_after_desuperheater"]
    Q_desuperheater = stream_data["Q_desuperheater"]

    # BOG 스트림 결과
    comp_bog = bog_data["comp_bog"]
    T_bog_comp_out = bog_data["T_bog_comp_out"]
    P_bog_comp_out = bog_data["P_bog_comp_out"]
    hpc = bog_data["hpc"]
    W_hpc = bog_data["W_hpc"]

    # 혼합 및 기화 결과
    T_bog_precooler_out = mixing_data["T_bog_precooler_out"]
    m_vaporizer_feed = mixing_data["m_vaporizer_feed"]
    T_vaporizer_feed = mixing_data["T_vaporizer_feed"]
    P_vaporizer_feed = mixing_data["P_vaporizer_feed"]
    T_after_reliq_mix = mixing_data["T_after_reliq_mix"]
    vap = mixing_data["vap"]

    # 총 전력 및 열량 계산
    total_power_mw = (pump["W"] + comp_bog["W"] + W_hpc) / 1e6
    vaporizer_duty_mw = vap["Q"] / 1e6
    pump_out_heating_duty_mw = Q_lng_pump_out_heating / 1e6
    desuperheater_duty_mw = Q_desuperheater / 1e6

    return {
        "BOG_TOTAL_MASS_FLOW": m_bog_total,              # 총 BOG 질량 유량
        "BOG_HPC_MASS_FLOW": m_bog_hpc,                  # HPC로 가는 BOG 질량 유량
        "BOG_PRECOOLER_MASS_FLOW": m_bog_pc,             # 프리쿨러로 가는 BOG 질량 유량

        "LNG_TANK_T_C": K_to_C(params["T_lng_tank"]), # LNG 탱크 온도 (섭씨)
        "LNG_TANK_P_BAR": Pa_to_bar(params["P_lng_tank"]), # LNG 탱크 압력 (bar)
        "LNG_PUMP_OUT_CALC_T_C": K_to_C(T_lng_after_pump_calc), # LNG 펌프 출구 계산된 온도 (섭씨)
        "LNG_PUMP_OUT_P_BAR": Pa_to_bar(P_lng_after_pump), # LNG 펌프 출구 압력 (bar)
        "LNG_PUMP_OUT_HEATED_T_C": K_to_C(T_lng_after_desuperheater), # LNG 펌프 출구 가열된 온도 (디슈퍼히터 후) (섭씨)
        "LNG_PUMP_OUT_HEATING_DUTY_MW": pump_out_heating_duty_mw, # LNG 펌프 출구 가열 열량 (MW)
        "BOG_RELIQ_RETURN_T_C": K_to_C(T_bog_precooler_out), # BOG 재액화 반환 온도 (섭씨)
        "AFTER_RELIQ_MIX_T_C": K_to_C(T_after_reliq_mix),   # 재액화 혼합 후 온도 (섭씨)
        "VAPORIZER_FEED_MASS_FLOW": m_vaporizer_feed,         # 기화기 공급 질량 유량
        "VAPORIZER_FEED_T_C": K_to_C(T_vaporizer_feed),       # 기화기 공급 온도 (섭씨)
        "VAPORIZER_FEED_P_BAR": Pa_to_bar(P_vaporizer_feed), # 기화기 공급 압력 (bar)
        "VAPORIZER_OUT_T_C": K_to_C(vap["T_ng_vap_out"]),   # 기화기 출구 온도 (섭씨)

        "BOG_TANK_T_C": K_to_C(params["T_bog_tank"]),     # BOG 탱크 온도 (섭씨)
        "BOG_TANK_P_BAR": Pa_to_bar(params["P_bog_tank"]), # BOG 탱크 압력 (bar)
        "BOG_COMP_OUT_T_C": K_to_C(T_bog_comp_out),       # BOG 압축기 출구 온도 (섭씨)
        "BOG_COMP_OUT_P_BAR": Pa_to_bar(P_bog_comp_out), # BOG 압축기 출구 압력 (bar)
        "BOG_HPC_IN_T_C": K_to_C(T_bog_hpc_in_after_desuperheater), # BOG HPC 입구 온도 (디슈퍼히터 후) (섭씨)
        "BOG_PRECOOLER_OUT_T_C": K_to_C(T_bog_precooler_out), # BOG 프리쿨러 출구 온도 (섭씨)
        "BOG_HPC_OUT_T_C": K_to_C(hpc["T_out"]),         # BOG HPC 출구 온도 (섭씨)
        "BOG_HPC_OUT_P_BAR": Pa_to_bar(hpc["P_out"]),    # BOG HPC 출구 압력 (bar)
        "BOG_AFTER_VALVE_T_C": K_to_C(vap["T_bog_after_valve"]), # BOG 밸브 통과 후 온도 (섭씨)
        "BOG_AFTER_VALVE_P_BAR": Pa_to_bar(P_sendout),   # BOG 밸브 통과 후 압력 (bar)

        "MIXED_SENDOUT_T_C": K_to_C(vap["T_mix_out"]),       # 혼합 송출 온도 (섭씨)
        "MIXED_SENDOUT_P_BAR": Pa_to_bar(P_sendout),         # 혼합 송출 압력 (bar)
        "MIXED_SENDOUT_MASS_FLOW": m_vaporizer_feed + m_bog_hpc, # 혼합 송출 질량 유량

        "DESUPERHEATER_DUTY_MW": desuperheater_duty_mw,    # 디슈퍼히터 열량 (MW)
        "VAPORIZER_DUTY_MW": vaporizer_duty_mw,            # 기화기 열량 (MW)
        "LNG_PUMP_POWER_MW": pump["W"] / 1e6,              # LNG 펌프 동력 (MW)
        "BOG_COMP_POWER_MW": comp_bog["W"] / 1e6,          # BOG 압축기 동력 (MW)
        "BOG_HPC_POWER_MW": W_hpc / 1e6,                   # BOG HPC 동력 (MW)
        "TOTAL_POWER_MW": total_power_mw,                  # 총 동력 (MW)

        "VAPORIZER_PLUS_POWER_MW": vaporizer_duty_mw + total_power_mw, # 기화기 열량 + 총 동력 (MW)
        "HEATING_PLUS_VAPORIZER_PLUS_POWER_MW": (Q_lng_pump_out_heating - Q_desuperheater) / 1e6 + vaporizer_duty_mw + total_power_mw, # 가열 + 기화기 열량 + 총 동력 (MW)

        "HPC_STAGE_RESULTS": hpc["stages"],             # HPC 단계별 결과
        "HPC_HX_RESULTS": [],                             # HPC 열교환기 결과 (현재 사용 안 함)
    }

def simulate_process(params):
    # 주요 매개변수 추출
    m_lng = params["m_lng"]
    m_bog_total = params["m_bog_total"]
    bog_to_hpc_frac = params["bog_to_hpc_frac"]

    # BOG 질량 유량 분배
    m_bog_hpc = m_bog_total * bog_to_hpc_frac
    m_bog_pc = m_bog_total - m_bog_hpc

    # 명확성을 위한 주요 매개변수 추출
    P_lng_tank = params["P_lng_tank"]
    T_lng_tank = params["T_lng_tank"]
    P_lng_high = params["P_lng_high"]
    T_lng_pump_out_initial = params["T_lng_pump_out_heated"]

    P_bog_tank = params["P_bog_tank"]
    T_bog_tank = params["T_bog_tank"]
    P_bog_comp_out = params["P_bog_comp_out"]
    P_bog_hpc_out = params["P_bog_hpc_out"]

    T_bog_reliq_return = params["T_bog_reliq_return_case1"]

    # 초기 BOG 압축기
    comp_bog_initial = compressor(
        m_dot=m_bog_total,
        T_in=T_bog_tank,
        P_in=P_bog_tank,
        P_out=P_bog_comp_out,
        eta=params["comp_eta"],
    )
    T_bog_comp_out_initial = comp_bog_initial["T_out"]
    P_bog_comp_out_initial = comp_bog_initial["P_out"]

    # 펌프, 초기 가열 및 디슈퍼히터 상호작용을 포함한 LNG 스트림 계산
    lng_stream_results = _calculate_lng_stream(
        params, m_lng, P_lng_tank, T_lng_tank, P_lng_high, T_lng_pump_out_initial,
        m_bog_hpc, T_bog_comp_out_initial, P_bog_comp_out_initial
    )

    # HPC를 포함한 BOG 스트림 계산
    bog_stream_results = _calculate_bog_streams(
        params, m_bog_total, m_bog_hpc, P_bog_tank, T_bog_tank, P_bog_comp_out,
        P_bog_hpc_out, lng_stream_results["T_bog_hpc_in_after_desuperheater"]
    )

    # 혼합 및 기화 계산
    mixing_vaporization_results = _calculate_mixing_and_vaporization(
        params, m_lng, m_bog_hpc, m_bog_pc, lng_stream_results["P_lng_after_pump"],
        lng_stream_results["T_lng_after_desuperheater"], T_bog_reliq_return,
        bog_stream_results["hpc"]["T_out"], bog_stream_results["hpc"]["P_out"]
    )

    # 최종 결과 취합
    return _assemble_results(
        params, lng_stream_results, bog_stream_results, mixing_vaporization_results
    )


# =====================================================
# 새로운 공정의 입력 조건
# =====================================================

params = {
    "m_lng": 65.0,

    # 총 BOG 유량, BOG 압축기 후 분할
    "m_bog_total": 35.0,
    "bog_to_hpc_frac": 0.60,

    # LNG 저장 탱크 및 실제 펌프 입구 조건
    "T_lng_tank": C_to_K(-160.0),
    "P_lng_tank": bar_to_Pa(5.0),

    # LNG 2차 펌프 출구는 펌핑 후 가열된다고 가정합니다.
    # 이것은 LNG 스트림의 디슈퍼히터 이전의 목표 온도입니다.
    "T_lng_pump_out_heated": C_to_K(-130.0),

    # LNG 2차 펌프 출구 압력
    "P_lng_high": bar_to_Pa(80.0),

    "T_bog_tank": C_to_K(-100.0),
    "P_bog_tank": bar_to_Pa(1.2),
    "P_bog_comp_out": bar_to_Pa(10.0),
    "P_bog_hpc_out": bar_to_Pa(80.0),

    "T_sendout": C_to_K(15.0),
    "P_sendout": bar_to_Pa(70.0),

    "pump_eta": 0.75,
    "comp_eta": 0.75,
    "desuperheater_effectiveness": 0.70, # 디슈퍼히터 효율에 대한 새로운 매개변수

    # Case 1 (기본) 별도 재액화 반환 조건.
    "T_bog_reliq_return_case1": C_to_K(-124.0),
}


In [54]:
import pandas as pd
from CoolProp.CoolProp import PropsSI

def get_phase_string(T_C, P_bar, fluid=FLUID):
    T_K = C_to_K(T_C)
    P_Pa = bar_to_Pa(P_bar)
    try:
        raw_phase_output = PropsSI('Phase', 'T', T_K, 'P', P_Pa, fluid)

        phase_name_str = None

        if isinstance(raw_phase_output, (int, float)): # 정수 또는 실수 형태의 위상 코드를 처리
            phase_code = int(raw_phase_output) # 정수형으로 변환하여 매핑
            # CoolProp의 표준 위상 이름 문자열로 매핑합니다.
            if phase_code == 0:
                phase_name_str = 'undefined'
            elif phase_code == 1:
                phase_name_str = 'liquid'
            elif phase_code == 2:
                phase_name_str = 'gas'
            elif phase_code == 3:
                phase_name_str = 'supercritical'
            elif phase_code == 4:
                phase_name_str = 'supercritical_gas'
            elif phase_code == 5:
                phase_name_str = 'supercritical_liquid'
            elif phase_code == 6:
                phase_name_str = 'critical_point'
            elif phase_code == 7:
                phase_name_str = 'two_phase'
            else:
                phase_name_str = f'알 수 없는 위상 코드: {phase_code}'
        elif isinstance(raw_phase_output, str):
            phase_name_str = raw_phase_output
        else:
            # 예측하지 못한 다른 타입의 출력인 경우
            phase_name_str = '알 수 없음 (CoolProp 출력 형식 오류 - 타입 미확인)'

        # 매핑된 문자열 위상 이름을 한국어로 번역
        if phase_name_str == 'liquid':
            return '액체'
        elif phase_name_str == 'gas':
            return '기체'
        elif phase_name_str in ['supercritical', 'supercritical_gas', 'supercritical_liquid']:
            return '초임계' # 초임계 관련 위상들을 하나로 묶어 처리
        elif phase_name_str == 'critical_point':
            return '임계점'
        elif phase_name_str == 'two_phase':
            Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
            return f'기액 이중상 (건도: {Q:.2f})'
        elif phase_name_str == 'undefined':
            # 'undefined'인 경우, 가능하면 건도를 통해 추가 추정
            try:
                Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
                if Q is not None:
                    if Q <= 0: return '과냉각 액체 또는 포화 액체'
                    elif Q >= 1: return '과열 증기 또는 포화 증기'
                    else: return f'기액 이중상 (건도: {Q:.2f})'
            except ValueError:
                pass # Quality 계산 실패 시 무시
            return '정의되지 않음'
        else:
            return phase_name_str # CoolProp이 반환하는 기타 문자열 위상 이름

    except ValueError:
        # CoolProp 계산 오류 발생 시 (예: 정의되지 않은 영역), 건도를 통해 추가 추정
        try:
            Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
            if Q is not None:
                if Q <= 0: return '과냉각 액체 또는 포화 액체'
                elif Q >= 1: return '과열 증기 또는 포화 증기'
                else: return f'기액 이중상 (건도: {Q:.2f})'
        except Exception:
            pass # Quality 계산 실패 시 무시
        return '알 수 없음 (CoolProp 오류)'
    except Exception as e:
        return f'오류 발생: {type(e).__name__}'

# 데이터 추출 및 가공
data_points = []

# --- LNG 스트림 ---
data_points.append({
    "위치": "LNG 탱크",
    "온도 [°C]": results["LNG_TANK_T_C"],
    "압력 [bar]": results["LNG_TANK_P_BAR"],
    "질량 유량 [kg/s]": params["m_lng"],
    "위상": get_phase_string(results["LNG_TANK_T_C"], results["LNG_TANK_P_BAR"])
})

data_points.append({
    "위치": "LNG 펌프 출구 (계산)",
    "온도 [°C]": results["LNG_PUMP_OUT_CALC_T_C"],
    "압력 [bar]": results["LNG_PUMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": params["m_lng"],
    "위상": get_phase_string(results["LNG_PUMP_OUT_CALC_T_C"], results["LNG_PUMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "LNG 펌프 출구 (초기 가열 목표)", # 디슈퍼히터 전 가열 목표 온도
    "온도 [°C]": K_to_C(params["T_lng_pump_out_heated"]),
    "압력 [bar]": results["LNG_PUMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": params["m_lng"],
    "위상": get_phase_string(K_to_C(params["T_lng_pump_out_heated"]), results["LNG_PUMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "LNG 디슈퍼히터 출구",
    "온도 [°C]": results["LNG_PUMP_OUT_HEATED_T_C"],
    "압력 [bar]": results["LNG_PUMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": params["m_lng"],
    "위상": get_phase_string(results["LNG_PUMP_OUT_HEATED_T_C"], results["LNG_PUMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "재액화 혼합점",
    "온도 [°C]": results["AFTER_RELIQ_MIX_T_C"],
    "압력 [bar]": results["LNG_PUMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": results["VAPORIZER_FEED_MASS_FLOW"],
    "위상": get_phase_string(results["AFTER_RELIQ_MIX_T_C"], results["LNG_PUMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "기화기 공급",
    "온도 [°C]": results["VAPORIZER_FEED_T_C"],
    "압력 [bar]": results["VAPORIZER_FEED_P_BAR"],
    "질량 유량 [kg/s]": results["VAPORIZER_FEED_MASS_FLOW"],
    "위상": get_phase_string(results["VAPORIZER_FEED_T_C"], results["VAPORIZER_FEED_P_BAR"])
})

data_points.append({
    "위치": "기화기 출구 (NG 스트림)",
    "온도 [°C]": results["VAPORIZER_OUT_T_C"],
    "압력 [bar]": results["MIXED_SENDOUT_P_BAR"],
    "질량 유량 [kg/s]": params["m_lng"],
    "위상": get_phase_string(results["VAPORIZER_OUT_T_C"], results["MIXED_SENDOUT_P_BAR"])
})


# --- BOG 스트림 ---
data_points.append({
    "위치": "BOG 탱크",
    "온도 [°C]": results["BOG_TANK_T_C"],
    "압력 [bar]": results["BOG_TANK_P_BAR"],
    "질량 유량 [kg/s]": params["m_bog_total"],
    "위상": get_phase_string(results["BOG_TANK_T_C"], results["BOG_TANK_P_BAR"])
})

data_points.append({
    "위치": "BOG 저압 압축기 출구",
    "온도 [°C]": results["BOG_COMP_OUT_T_C"],
    "압력 [bar]": results["BOG_COMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": params["m_bog_total"],
    "위상": get_phase_string(results["BOG_COMP_OUT_T_C"], results["BOG_COMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "BOG 프리쿨러 출구 (재액화 반환)",
    "온도 [°C]": results["BOG_RELIQ_RETURN_T_C"],
    "압력 [bar]": results["LNG_PUMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": results["BOG_PRECOOLER_MASS_FLOW"],
    "위상": get_phase_string(results["BOG_RELIQ_RETURN_T_C"], results["LNG_PUMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "BOG HPC 입구 (디슈퍼히터 출구)",
    "온도 [°C]": results["BOG_HPC_IN_T_C"],
    "압력 [bar]": results["BOG_COMP_OUT_P_BAR"],
    "질량 유량 [kg/s]": results["BOG_HPC_MASS_FLOW"],
    "위상": get_phase_string(results["BOG_HPC_IN_T_C"], results["BOG_COMP_OUT_P_BAR"])
})

data_points.append({
    "위치": "BOG HPC 출구",
    "온도 [°C]": results["BOG_HPC_OUT_T_C"],
    "압력 [bar]": results["BOG_HPC_OUT_P_BAR"],
    "질량 유량 [kg/s]": results["BOG_HPC_MASS_FLOW"],
    "위상": get_phase_string(results["BOG_HPC_OUT_T_C"], results["BOG_HPC_OUT_P_BAR"])
})

data_points.append({
    "위치": "BOG 밸브 통과 후",
    "온도 [°C]": results["BOG_AFTER_VALVE_T_C"],
    "압력 [bar]": results["BOG_AFTER_VALVE_P_BAR"],
    "질량 유량 [kg/s]": results["BOG_HPC_MASS_FLOW"],
    "위상": get_phase_string(results["BOG_AFTER_VALVE_T_C"], results["BOG_AFTER_VALVE_P_BAR"])
})

# --- 최종 생산물 스트림 ---
data_points.append({
    "위치": "최종 송출 가스",
    "온도 [°C]": results["MIXED_SENDOUT_T_C"],
    "압력 [bar]": results["MIXED_SENDOUT_P_BAR"],
    "질량 유량 [kg/s]": results["MIXED_SENDOUT_MASS_FLOW"],
    "위상": get_phase_string(results["MIXED_SENDOUT_T_C"], results["MIXED_SENDOUT_P_BAR"])
})

df_streams_full = pd.DataFrame(data_points)

print("\n=== 전체 시스템 스트림 데이터 요약 ===")
display(df_streams_full.round(2).fillna('N/A'))


=== 전체 시스템 스트림 데이터 요약 ===


,위치,온도 [°C],압력 [bar],질량 유량 [kg/s],위상
0,LNG 탱크,-160.00,5.0,65.0,과냉각 액체 또는 포화 액체
1,LNG 펌프 출구 (계산),-131.49,80.0,65.0,초임계
2,LNG 펌프 출구 (초기 가열 목표),-130.00,80.0,65.0,초임계
3,LNG 디슈퍼히터 출구,-106.50,80.0,65.0,초임계
4,재액화 혼합점,-109.40,80.0,79.0,초임계
5,기화기 공급,-109.40,80.0,79.0,초임계
6,기화기 출구 (NG 스트림),-9.40,70.0,65.0,액체
7,BOG 탱크,-100.00,1.2,35.0,초임계
8,BOG 저압 압축기 출구,61.28,10.0,35.0,기체
9,BOG 프리쿨러 출구 (재액화 반환),-124.00,80.0,14.0,초임계


## 스트림 데이터 상세 정보 (값, 유형 및 연결)

다음 표는 각 스트림 위치의 온도, 압력, 질량 유량에 대한 실제 값과 해당 값이 '입력' 값인지 '계산'된 값인지, 그리고 유체의 다음 연결 위치를 상세히 보여줍니다.

| 위치                           | 온도 [°C] | 압력 [bar] | 질량 유량 [kg/s] | 다음 위치                                                        |
| :----------------------------- | :-------- | :--------- | :--------------- | :--------------------------------------------------------------- |
| LNG 탱크                       | -160.00 (입력) | 5.00 (입력) | 65.00 (입력)     | LNG 펌프 출구 (계산)                                             |
| LNG 펌프 출구 (계산)           | -131.49 (계산) | 80.00 (입력) | 65.00 (입력)     | LNG 펌프 출구 (초기 가열 목표)                                   |
| LNG 펌프 출구 (초기 가열 목표) | -130.00 (입력) | 80.00 (입력) | 65.00 (입력)     | LNG 디슈퍼히터 출구                                              |
| LNG 디슈퍼히터 출구            | -106.50 (계산) | 80.00 (입력) | 65.00 (입력)     | 재액화 혼합점                                                    |
| 재액화 혼합점                  | -109.40 (계산) | 80.00 (입력) | 79.00 (계산)     | 기화기 공급                                                      |
| 기화기 공급                    | -109.40 (계산) | 80.00 (입력) | 79.00 (계산)     | 기화기 출구 (NG 스트림)                                          |
| 기화기 출구 (NG 스트림)        | -9.40 (계산) | 70.00 (입력) | 65.00 (입력)     | 최종 송출 가스                                                   |
| BOG 탱크                       | -100.00 (입력) | 1.20 (입력) | 35.00 (입력)     | BOG 저압 압축기 출구                                             |
| BOG 저압 압축기 출구           | 61.28 (계산) | 10.00 (입력) | 35.00 (입력)     | BOG 프리쿨러 출구 (재액화 반환) 및 BOG HPC 입구 (디슈퍼히터 출구) |
| BOG 프리쿨러 출구 (재액화 반환) | -124.00 (입력) | 80.00 (입력) | 14.00 (계산)     | 재액화 혼합점                                                    |
| BOG HPC 입구 (디슈퍼히터 출구) | -73.59 (계산) | 10.00 (입력) | 21.00 (계산)     | BOG HPC 출구                                                     |
| BOG HPC 출구                   | 113.59 (계산) | 80.00 (입력) | 21.00 (계산)     | BOG 밸브 통과 후                                                 |
| BOG 밸브 통과 후               | 111.83 (계산) | 70.00 (입력) | 21.00 (계산)     | 최종 송출 가스                                                   |
| 최종 송출 가스                 | 15.00 (입력) | 70.00 (입력) | 100.00 (계산)    | 시스템 외부                                                      |
